# 🔗 Workflow Integration - Complete SAR Processing System

Welcome to Phase 4 of the Financial Services Agentic AI Project!

In this notebook, you'll integrate both AI agents into a complete **end-to-end SAR processing workflow** that demonstrates real-world financial compliance automation.

## 🎯 Learning Objectives
- Build a complete two-stage AI workflow with human oversight
- Implement human-in-the-loop decision gates for compliance
- Generate complete SAR documents from AI analysis
- Create comprehensive audit trails for regulatory examination
- Demonstrate cost optimization through intelligent agent coordination

## 📋 Business Context
This workflow simulates how banks actually process suspicious activity reports:
1. **Risk Screening**: AI agents analyze transaction patterns for suspicious activity
2. **Human Review**: Compliance officers review AI findings before proceeding
3. **Narrative Generation**: Only approved cases get full compliance documentation
4. **SAR Filing**: Complete regulatory forms are generated for submission
5. **Audit Documentation**: Every decision is logged for regulatory examination

## 🏗️ System Architecture

```
📊 CSV Data → 🔍 Risk Analyst → 👤 Human Decision → ✅ Compliance Officer → 📄 SAR Document
              (Chain-of-Thought)    (Gate)         (ReACT Framework)     (FinCEN Ready)
```

## 🚀 Prerequisites Check

Before starting, ensure you have completed:
- ✅ Phase 1: Foundation components (`foundation_sar.py`)
- ✅ Phase 2: Risk Analyst Agent (`risk_analyst_agent.py`)
- ✅ Phase 3: Compliance Officer Agent (`compliance_officer_agent.py`)
- ✅ Both agents pass their comprehensive test scenarios

If any component is missing, return to previous notebooks to complete implementation.

In [1]:
# Setup and Environment Configuration
import os
import sys
import json
import pandas as pd
import uuid
import hashlib
from datetime import datetime, timedelta
from dotenv import load_dotenv

# Project paths: src/ for the modules, the project root for the `src` package helpers
project_root = os.path.abspath('..')
src_path = os.path.join(project_root, 'src')
for path in (src_path, project_root):
    if path not in sys.path:
        sys.path.insert(0, path)

load_dotenv(os.path.join(project_root, '.env'))
MODEL = os.getenv('OPENAI_MODEL', 'gpt-4o-mini')

# Human review configuration:
#   'interactive'      - you approve/reject each case at both gates (default when you run the notebook)
#   'simulated_policy' - a documented reviewer policy decides (for unattended runs, e.g. nbconvert)
REVIEW_MODE = os.getenv('SAR_REVIEW_MODE', 'interactive')
REVIEWER_ID = os.getenv('SAR_REVIEWER_ID', 'compliance_reviewer_01')
CASE_LIMIT = None if REVIEW_MODE == 'simulated_policy' else 5   # keep interactive sessions short

RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
AUDIT_LOG_PATH = os.path.join(project_root, 'outputs', 'audit_logs', f'sar_workflow_{RUN_ID}.jsonl')
METRICS_PATH = os.path.join(project_root, 'outputs', 'audit_logs', f'workflow_metrics_{RUN_ID}.json')
SAR_OUTPUT_DIR = os.path.join(project_root, 'outputs', 'filed_sars')

pd.set_option('display.width', 220)
pd.set_option('display.max_colwidth', 90)

print("📚 Libraries imported successfully!")
print("🔐 Environment variables loaded")
print("📂 Source directory added to Python path")
print(f"🤖 Model: {MODEL} | 👤 Review mode: {REVIEW_MODE} (reviewer {REVIEWER_ID}) | 🆔 Run: {RUN_ID}")

📚 Libraries imported successfully!
🔐 Environment variables loaded
📂 Source directory added to Python path
🤖 Model: gpt-4o-mini | 👤 Review mode: interactive (reviewer compliance_reviewer_01) | 🆔 Run: 20261011_131942


In [2]:
# OpenAI Setup - Vocareum endpoint for voc- keys (Udacity workspace), standard OpenAI otherwise
from src import create_openai_client

client = create_openai_client()

✅ OpenAI client initialized (OpenAI key)
📍 Base URL: https://api.openai.com/v1 (default)


In [3]:
# Import the implemented components and wire up the workflow
from foundation_sar import (
    CustomerData,
    AccountData,
    TransactionData,
    CaseData,
    RiskAnalystOutput,
    ComplianceOfficerOutput,
    ExplainabilityLogger,
    DataLoader,
    load_csv_data
)
from risk_analyst_agent import RiskAnalystAgent
from compliance_officer_agent import ComplianceOfficerAgent
import sar_workflow
from sar_workflow import (
    screen_customers, HumanReviewGate, SARWorkflow, CaseStatus,
    verify_sar_document, compute_workflow_metrics
)

# One audit trail for the whole run: DataLoader, RiskAnalyst, HumanReviewer, ComplianceOfficer, SARFiling, Workflow
explainability_logger = ExplainabilityLogger(AUDIT_LOG_PATH)
risk_agent = RiskAnalystAgent(client, explainability_logger, model=MODEL)
compliance_agent = ComplianceOfficerAgent(client, explainability_logger, model=MODEL)
review_gate = HumanReviewGate(explainability_logger, reviewer_id=REVIEWER_ID, mode=REVIEW_MODE)
workflow = SARWorkflow(risk_agent, compliance_agent, review_gate, DataLoader(explainability_logger),
                       explainability_logger, sar_output_dir=SAR_OUTPUT_DIR, run_id=RUN_ID)

print("✅ Components imported and workflow assembled")
print(f"📝 Audit log:  {os.path.relpath(AUDIT_LOG_PATH, project_root)}")
print(f"📄 SAR output: {os.path.relpath(SAR_OUTPUT_DIR, project_root)}/")

✅ Components imported and workflow assembled
📝 Audit log:  outputs/audit_logs/sar_workflow_20261011_131942.jsonl
📄 SAR output: outputs/filed_sars/


## 📊 Step 1: Data Loading and Preprocessing

Load the financial data and prepare it for analysis.

In [4]:
# Load and Preprocess Financial Data

def load_and_preprocess_data():
    """Load the CSV extracts and convert them to record dictionaries

    No manual NaN handling is needed: the Pydantic schemas turn empty cells
    into None for optional fields and restore ssn_last_4 as a 4-digit string.

    Returns:
        (customers_data, accounts_data, transactions_data) as lists of dicts
    """
    print("📊 Loading Financial Data")
    customers_df, accounts_df, transactions_df = load_csv_data(os.path.join(project_root, 'data'))

    customers_data = customers_df.to_dict('records')
    accounts_data = accounts_df.to_dict('records')
    transactions_data = transactions_df.to_dict('records')

    print(f"📈 Loaded: {len(customers_data)} customers, {len(accounts_data)} accounts, "
          f"{len(transactions_data):,} transactions")
    print(f"🗓️  Activity period: {transactions_df['transaction_date'].min()} to {transactions_df['transaction_date'].max()}")
    return customers_data, accounts_data, transactions_data

# Load data
customers_data, accounts_data, transactions_data = load_and_preprocess_data()

📊 Loading Financial Data
📈 Loaded: 150 customers, 178 accounts, 4,268 transactions
🗓️  Activity period: 2025-02-03 to 2025-08-06


## 🎯 Step 2: Customer Risk Screening

Implement intelligent customer screening to identify high-risk cases for detailed analysis.

In [5]:
# Customer Risk Screening

def screen_high_risk_customers(customers_data, accounts_data, transactions_data, top_n=None):
    """Score every customer and return the alert queue for AI analysis

    Screening signals (points):
    1. Typology rules validated in Phase 1 - structuring (40), rapid movement (40),
       large cash (30), cash velocity across branches (25)
    2. Customer risk rating - High (15), Medium (5)
    3. Activity volume above 2x reported income (10)
    4. Activity on closed accounts (10)

    Customers scoring >= 15 raise an alert. Profile-only alerts are kept on purpose:
    real monitoring queues contain false positives, which the Risk Analyst and the
    drafting gate are there to filter out.

    Returns the top N alerts (all alerts when top_n is None), highest score first.
    """
    global screening_results
    print("🔍 Customer Risk Screening")
    screening_results = screen_customers(pd.DataFrame(customers_data), pd.DataFrame(accounts_data),
                                         pd.DataFrame(transactions_data))
    alerts = screening_results[screening_results['alert']]
    selected = alerts if top_n is None else alerts.head(top_n)

    print(f"📊 {len(screening_results)} customers with activity screened -> {len(alerts)} alerts "
          f"({alerts['typology_signals'].apply(bool).sum()} with typology signals, "
          f"{(~alerts['typology_signals'].apply(bool)).sum()} profile-only)")
    display_table = selected[['customer_id', 'name', 'risk_rating', 'screening_score', 'signals']].copy()
    display_table['signals'] = display_table['signals'].apply(', '.join)
    print(display_table.to_string(index=False))
    print(f"\n📋 Selected {len(selected)} customers for two-stage analysis")
    return selected.to_dict('records')

# Run customer screening
selected_customers = screen_high_risk_customers(customers_data, accounts_data, transactions_data, top_n=CASE_LIMIT)

🔍 Customer Risk Screening
📊 119 customers with activity screened -> 25 alerts (21 with typology signals, 4 profile-only)
customer_id              name risk_rating  screening_score                                                     signals
  CUST_0018    Tanya Johnston        High               65      structuring, high_risk_rating, closed_account_activity
  CUST_0081 Matthew Alexander      Medium               60               large_cash, cash_velocity, medium_risk_rating
  CUST_0009     Brandi Bailey      Medium               55 rapid_movement, medium_risk_rating, closed_account_activity
  CUST_0053       Tracy Lewis      Medium               55           structuring, medium_risk_rating, volume_vs_income
  CUST_0062   Michael Stanley      Medium               55        rapid_movement, medium_risk_rating, volume_vs_income

📋 Selected 5 customers for two-stage analysis


## 🤖 Step 3: Two-Stage AI Analysis with Human Gates

Each alerted customer goes through:
1. **Stage 1:** the Risk Analyst performs Chain-of-Thought analysis.
2. **👤 Gate 1 (SAR drafting):** a reviewer approves or rejects drafting. Rejections close the alert with a documented "no SAR" rationale, and **no compliance call is made**.
3. **Stage 2:** the Compliance Officer drafts the ReACT narrative (approved cases only).
4. **👤 Gate 2 (SAR filing):** a reviewer approves the narrative for filing, or returns it for revision.
5. **Filing:** a checksummed SAR document is written to `outputs/filed_sars/`.

Every step, including each human decision, is written to the run's audit log.

**Review mode.** Run this notebook yourself with `SAR_REVIEW_MODE=interactive` (the default) and you make every decision at the prompts. The review card shows the AI findings plus the documented policy's guidance. For unattended execution, `SAR_REVIEW_MODE=simulated_policy` applies that policy instead:
- **Gate 1:** escalate High/Critical findings with confidence ≥ 0.70, and all sanctions exposure.
- **Gate 2:** file only complete narratives with no accuracy or tone warnings.

Every decision records its `decision_source` (`interactive` or `simulated_policy`), so simulated decisions can never be mistaken for human ones in the audit trail.

In [6]:
# Two-Stage AI Workflow

def run_two_stage_sar_workflow(selected_customers):
    """Run every selected customer through Risk Analyst -> gate 1 -> Compliance Officer -> gate 2 -> filing

    Returns:
        processed_cases: CaseRecord per customer (full state, see sar_workflow.CaseRecord)
        approved_sars: SAR documents filed in this run
        rejected_cases: Cases not filed, with the stage and rationale
        audit_decisions: One row per human decision
    """
    print(f"🤖 Two-Stage SAR Processing Workflow - {len(selected_customers)} cases, review mode '{REVIEW_MODE}'")
    print("=" * 100)
    customer_lookup = {customer['customer_id']: customer for customer in customers_data}
    progress = print if REVIEW_MODE == 'simulated_policy' else None  # interactive mode prints review cards instead
    processed_cases = workflow.run(selected_customers, customer_lookup, accounts_data, transactions_data,
                                   progress=progress)

    approved_sars = []
    for record in processed_cases:
        if record.status == CaseStatus.FILED:
            with open(record.sar_path) as f:
                approved_sars.append(json.load(f))

    rejected_cases = [
        {
            'case_id': record.case_id,
            'customer_id': record.customer_id,
            'stage': record.status.value,
            'reason': {CaseStatus.REJECTED_AT_DRAFTING: lambda r: r.drafting_decision.rationale,
                       CaseStatus.REJECTED_AT_FILING: lambda r: r.filing_decision.rationale}
                      .get(record.status, lambda r: r.error)(record)
        }
        for record in processed_cases if record.status != CaseStatus.FILED
    ]

    audit_decisions = [
        {
            'case_id': record.case_id,
            'customer_name': record.customer_name,
            'gate': decision.gate,
            'decision': decision.decision.upper(),
            'ai_classification': record.risk_analysis.classification,
            'ai_risk_level': record.risk_analysis.risk_level,
            'ai_confidence': record.risk_analysis.confidence_score,
            'decision_source': decision.decision_source,
            'reviewer_decision': decision.rationale
        }
        for record in processed_cases
        for decision in (record.drafting_decision, record.filing_decision) if decision
    ]

    print("=" * 100)
    print(f"✅ SARs filed: {len(approved_sars)} | ❌ Not filed: {len(rejected_cases)} | "
          f"👤 Human decisions: {len(audit_decisions)}")
    return processed_cases, approved_sars, rejected_cases, audit_decisions

# Run the complete workflow
processed_cases, approved_sars, rejected_cases, audit_decisions = run_two_stage_sar_workflow(selected_customers)

🤖 Two-Stage SAR Processing Workflow - 5 cases, review mode 'interactive'
══════════════════════════════════════════════════════════════════════════════
👤 HUMAN REVIEW · GATE 1 - Approve SAR drafting
══════════════════════════════════════════════════════════════════════════════
Case:        c44e491c-0091-4117-8343-0162b74af83f
Subject:     Tanya Johnston (CUST_0018) · Therapist, drama · income $166,876 · internal rating High
Activity:    54 transactions, 2025-02-08 to 2025-07-27, 2 account(s)
AI finding:  Structuring · High risk · confidence 0.85
Indicators:  - 4 cash deposits of $9k-$9.6k within 4 days
             - Deposits made at 3 different branches
             - Transactions posted to a closed account
             - Contradictory transaction descriptions
Reasoning:   Profile: Tanya is a therapist with a high income and risk rating. Pattern: 4 cash deposits between $9k-$10k totaling $37,536.45 in 4 days. Mapping: This suggests structuring to evade CTR reporting. Risk: High due to

KeyboardInterrupt: Interrupted by user

In [ ]:
# Case outcomes and the documented human decisions
outcome_table = pd.DataFrame([
    {
        'customer': f"{r.customer_id} {r.customer_name}",
        'screening': r.screening_score,
        'ai_finding': f"{r.risk_analysis.classification}/{r.risk_analysis.risk_level}" if r.risk_analysis else '-',
        'confidence': r.risk_analysis.confidence_score if r.risk_analysis else None,
        'gate_1': r.drafting_decision.decision if r.drafting_decision else '-',
        'words': (r.narrative_validation or {}).get('word_count', '-') if r.compliance_output else '-',
        'gate_2': r.filing_decision.decision if r.filing_decision else '-',
        'outcome': r.status.value
    }
    for r in processed_cases
])
print("📋 Case Outcomes")
print(outcome_table.to_string(index=False))

print("\n🚫 Cases not filed - documented rationale:")
for case in rejected_cases:
    print(f"   • {case['customer_id']} [{case['stage']}]: {case['reason']}")

## 📄 Step 4: SAR Document Generation

Create complete, FinCEN-ready SAR documents with all required metadata.

In [ ]:
# SAR Document Generation

def create_sar_document(case_data, risk_analysis, compliance_review, record=None):
    """Create a complete SAR document

    The document (see sar_workflow.create_sar_document) contains:
    1. SAR metadata - ID, preparation date, detection date, 30-day filing deadline, SHA-256 checksum
    2. Filing institution information
    3. Subject information - identity, masked TIN, accounts
    4. Suspicious activity - classification, period, flagged transactions and total, indicators
    5. Compliance narrative with word count, citations and validation warnings
    6. AI analysis summaries (Chain-of-Thought and ReACT)
    7. Both human review decisions, and a pointer to the audit trail
    """
    record = record or next(r for r in processed_cases if r.case_id == case_data.case_id)
    record = record.model_copy(update={'risk_analysis': risk_analysis, 'compliance_output': compliance_review})
    flagged_ids = screening_results.set_index('customer_id').loc[record.customer_id, 'flagged_transaction_ids']
    return sar_workflow.create_sar_document(record, case_data, RUN_ID, explainability_logger.log_file, flagged_ids)

def save_sar_document(sar_document):
    """Save SAR document to outputs/filed_sars/<sar_id>.json"""
    return sar_workflow.save_sar_document(sar_document, SAR_OUTPUT_DIR)

print("📄 SAR document generation functions defined")

# The workflow above already created and saved a SAR for every case approved at both gates
print(f"\n📁 SAR documents filed in this run ({len(approved_sars)}):")
for record in processed_cases:
    if record.status == CaseStatus.FILED:
        print(f"   {'✅' if verify_sar_document(record.sar_path) else '❌'} {os.path.basename(record.sar_path)} "
              f"- {record.risk_analysis.classification}, checksum verified")

if approved_sars:
    example = json.loads(json.dumps(approved_sars[0]))
    flagged = example['suspicious_activity']['flagged_transactions']
    example['suspicious_activity']['flagged_transactions'] = flagged[:2] + (
        [f"... {len(flagged) - 2} more flagged transactions"] if len(flagged) > 2 else [])
    example['subject_information']['accounts'] = example['subject_information']['accounts'][:1]
    print("\n🔍 Example SAR document (flagged transactions and accounts truncated for display):")
    print(json.dumps(example, indent=2))

## 📊 Step 5: Workflow Metrics and Analysis

Analyze the efficiency and effectiveness of your AI-powered SAR processing system.

In [ ]:
# Workflow Analysis and Metrics

def analyze_workflow_efficiency(processed_cases, approved_sars, rejected_cases, audit_decisions):
    """Calculate and report workflow efficiency metrics, and save them next to the audit log

    1. Processing funnel and efficiency (time per case and per stage)
    2. Cost optimisation: actual vs single-stage vs no-screening cost
    3. Human decision patterns
    4. AI accuracy validation
    5. Regulatory compliance rates
    """
    with open(explainability_logger.log_file) as f:
        audit_entries = [json.loads(line) for line in f]
    metrics = compute_workflow_metrics(processed_cases, audit_entries,
                                       customers_with_activity=len(screening_results),
                                       alerts_raised=int(screening_results['alert'].sum()))

    funnel, costs, timing = metrics['funnel'], metrics['cost_optimization'], metrics['timing']
    print("📊 Workflow Efficiency Analysis")
    print("=" * 70)
    print("📈 WORKFLOW FUNNEL:")
    print(f"   Customers screened (with activity): {funnel['customers_with_activity']}")
    print(f"   Alerts raised / processed:          {funnel['alerts_raised']} / {funnel['alerts_processed']}")
    print(f"   Risk analyses (Stage 1):            {funnel['risk_analyses']}")
    print(f"   Rejected at drafting gate:          {funnel['rejected_at_drafting_gate']}")
    print(f"   Narratives generated (Stage 2):     {funnel['narratives_generated']}")
    print(f"   Rejected at filing gate:            {funnel['rejected_at_filing_gate']}")
    print(f"   SARs filed:                         {funnel['sars_filed']}")
    print(f"   Narrative failures / errors:        {funnel['narrative_failures']} / {funnel['errors']}")

    print("\n⏱️  PROCESSING EFFICIENCY:")
    stage_table = pd.DataFrame([
        {'stage': name, 'runs': s['runs'], 'api_calls': s['api_calls'], 'avg_latency_s': round(s['avg_latency_s'], 1),
         'tokens': s['prompt_tokens'] + s['completion_tokens'], 'cost_usd': round(s['cost_usd'], 5),
         'avg_cost_usd': round(s['avg_cost_usd'], 5)}
        for name, s in metrics['stage_costs'].items()
    ])
    print(stage_table.to_string(index=False))
    print(f"   Average time per case: {timing['avg_case_time_s']:.1f} s | total processing: {timing['total_processing_time_s']:.0f} s")

    print("\n💰 COST OPTIMIZATION (two-stage vs alternatives):")
    print(f"   Actual cost (screening + gates):             ${costs['actual_cost_usd']:.4f}")
    print(f"   Single-stage (narrative for every analysis): ${costs['single_stage_cost_usd']:.4f} "
          f"-> {costs['savings_vs_single_stage_pct']:.1f}% saved, {costs['compliance_runs_avoided_by_gate']} compliance runs avoided")
    print(f"   No screening (both agents on every customer): ${costs['no_screening_cost_usd']:.4f} "
          f"-> {costs['savings_vs_no_screening_pct']:.1f}% saved, {costs['risk_analyses_avoided_by_screening']} analyses avoided")
    if costs['cost_per_filed_sar_usd']:
        print(f"   Cost per filed SAR: ${costs['cost_per_filed_sar_usd']:.4f}")

    metrics['run_id'] = RUN_ID
    metrics['review_mode'] = REVIEW_MODE
    metrics['case_outcomes'] = outcome_table.to_dict('records')
    with open(METRICS_PATH, 'w') as f:
        json.dump(metrics, f, indent=2, default=str)
    print(f"\n💾 Metrics saved to {os.path.relpath(METRICS_PATH, project_root)}")
    return metrics

def validate_ai_decisions(audit_decisions):
    """Analyze AI decision patterns: accuracy, confidence and human decision patterns"""
    quality = workflow_metrics['ai_quality']
    print("\n🧠 AI Decision Validation")
    print("=" * 70)
    agreement = quality['classification_agreement_with_screening']
    print(f"🎯 AI classification matches the screening typology in "
          f"{agreement:.0%} of {quality['cases_with_screening_expectation']} typology alerts")
    mismatches = [(r.customer_id, r.expected_classification, r.risk_analysis.classification)
                  for r in processed_cases if r.risk_analysis and r.expected_classification
                  and r.risk_analysis.classification != r.expected_classification]
    for customer_id, expected, actual in mismatches:
        print(f"   • {customer_id}: screening suggested {expected}, AI classified {actual}")
    print(f"📊 Confidence: mean {quality['confidence_mean']:.2f}, range "
          f"{quality['confidence_min']:.2f}-{quality['confidence_max']:.2f}")

    print("\n👤 Human decision patterns:")
    for gate, stats in workflow_metrics['human_decisions'].items():
        print(f"   {gate}: {stats['reviewed']} reviewed, {stats['approved']} approved, {stats['rejected']} rejected "
              f"({stats['approval_rate']:.0%} approval) - source: {', '.join(stats['decision_sources']) or '-'}")
    decisions_df = pd.DataFrame(audit_decisions)
    if not decisions_df.empty:
        print("\n   Decisions by AI risk level (gate 1):")
        gate1 = decisions_df[decisions_df['gate'] == 'sar_drafting']
        print("   " + gate1.groupby(['ai_risk_level', 'decision']).size().to_string().replace('\n', '\n   '))

    print("\n📝 Regulatory compliance of generated narratives:")
    print(f"   Complete: {quality['narratives_complete_pct']:.0f}% | within 120 words: "
          f"{quality['narratives_within_word_limit_pct']:.0f}% | avg {quality['avg_narrative_words']:.0f} words | "
          f"with reviewer warnings: {quality['narratives_with_warnings']}")

# Run analysis
workflow_metrics = analyze_workflow_efficiency(processed_cases, approved_sars, rejected_cases, audit_decisions)
validate_ai_decisions(audit_decisions)

## 🏁 Step 6: Complete System Demonstration

Two demonstrations of the finished system:
1. **Audit-trail walkthrough:** every logged step for one filed case and one rejected case, reconstructed from the JSONL log alone. This is what an examiner would follow.
2. **Interactive review gate:** the review card a human sees, plus input handling. Invalid answers are re-prompted, and a rejection is refused until a rationale is given. Scripted keystrokes are used here so it runs unattended; set `SAR_REVIEW_MODE=interactive` to make the real decisions in Step 3.

In [ ]:
# Complete system demonstration

def demonstrate_complete_system():
    """Reconstruct case histories from the audit log and demonstrate the interactive gate"""
    with open(explainability_logger.log_file) as f:
        audit_entries = [json.loads(line) for line in f]

    print("🏁 Complete SAR Processing System Demonstration")
    print("=" * 90)
    filed = next((r for r in processed_cases if r.status == CaseStatus.FILED), None)
    rejected = next((r for r in processed_cases if r.status == CaseStatus.REJECTED_AT_DRAFTING), None)
    for label, record in (("FILED", filed), ("REJECTED AT GATE 1", rejected)):
        if record is None:
            continue
        print(f"\n🔎 Audit trail - {label}: {record.customer_id} {record.customer_name} (case {record.case_id})")
        for entry in (e for e in audit_entries if e['case_id'] == record.case_id):
            print(f"   {entry['timestamp'][11:19]}  {entry['agent_type']:<18} {entry['action']:<20} "
                  f"{'✅' if entry['success'] else '❌'} {entry['reasoning'][:85]}")

    run_summary = next(e for e in audit_entries if e['action'] == 'run_summary')
    print(f"\n🧾 Run summary entry: {run_summary['output_summary']}")
    print(f"📄 SAR documents saved to: outputs/filed_sars/ | 📊 Audit log: {os.path.relpath(explainability_logger.log_file, project_root)}")

    if filed is not None:
        print("\n👤 Interactive review gate - scripted keystrokes: 'maybe' (invalid), 'r', '' (no rationale), rationale")
        keystrokes = iter(["maybe", "r", "", "Funds traced to documented business sale - no SAR"])
        demo_case = DataLoader(ExplainabilityLogger(os.devnull)).create_case_from_data(
            next(c for c in customers_data if c['customer_id'] == filed.customer_id), accounts_data, transactions_data)
        demo_gate = HumanReviewGate(ExplainabilityLogger(os.devnull), reviewer_id='demo_reviewer', mode='interactive',
                                    input_fn=lambda prompt: (print(prompt + "▸"), next(keystrokes))[1])
        decision = demo_gate.review_risk_analysis(demo_case, filed.risk_analysis)
        print(f"\n➡️  Recorded decision: {decision.decision.upper()} by {decision.reviewer_id} "
              f"({decision.decision_source}) - \"{decision.rationale}\"")

demonstrate_complete_system()

## 📝 Implementation Checklist

### ✅ Workflow Integration Deliverables
- [x] **Data Loading**: Load and preprocess CSV data with proper error handling
- [x] **Customer Screening**: Implement risk-based screening to identify high-risk cases
- [x] **Two-Stage Workflow**: Build complete Risk Analyst → Human Gate → Compliance Officer flow
- [x] **Human Decision Gates**: Implement interactive approval/rejection points
- [x] **SAR Document Generation**: Create complete FinCEN-ready documents with metadata
- [x] **Audit Trail Creation**: Log all decisions and reasoning for regulatory examination
- [x] **Efficiency Metrics**: Calculate cost optimization and processing efficiency
- [x] **System Demonstration**: Test complete workflow with multiple scenarios

### ✅ Testing and Validation Requirements
- [x] **Component Validation**: Verify all foundation components and agents are available
- [x] **Integration Testing**: Run comprehensive test suites for all components with proper sys.path setup
- [x] **End-to-End Testing**: Test complete workflow with automated scenarios
- [x] **Error Handling Testing**: Validate graceful handling of edge cases and failures
- [x] **Output Validation**: Ensure SAR documents meet regulatory standards
- [x] **Performance Testing**: Measure workflow efficiency and processing times

### ✅ Technical Requirements
- [x] **Error Handling**: Robust exception handling for all workflow steps
- [x] **Data Validation**: Proper validation of all inputs and outputs
- [x] **File Management**: Organize outputs in appropriate directories
- [x] **Logging**: Comprehensive audit logging for compliance
- [x] **Performance**: Efficient processing of multiple cases
- [x] **User Experience**: Clear prompts and feedback for human reviewers
- [x] **Test Infrastructure**: Proper test imports and sys.path configuration

### ✅ Business Requirements  
- [x] **Regulatory Compliance**: Ensure all SAR documents meet FinCEN requirements
- [x] **Cost Optimization**: Demonstrate savings from two-stage processing
- [x] **Audit Readiness**: Create examination-ready documentation
- [x] **Quality Assurance**: Validate AI decisions with human oversight
- [x] **Scalability**: Design for processing larger datasets
- [x] **Production Readiness**: Complete testing validates system reliability

## 🎯 Success Criteria

By completion, your integrated system should:
- ✅ Process real financial data with proper validation
- ✅ Execute complete two-stage AI workflow with human gates
- ✅ Generate regulatory-compliant SAR documents
- ✅ Create comprehensive audit trails for all decisions
- ✅ Demonstrate measurable cost optimization benefits
- ✅ Handle errors gracefully and provide clear user feedback
- ✅ Pass all integration and end-to-end tests
- ✅ Meet production-ready quality standards

## 🚀 Next Steps

1. **Complete Implementation**: Fill in all TODO sections with working code
2. **Run Integration Tests**: Validate all components work together properly
3. **Execute End-to-End Tests**: Test complete workflow with automated scenarios
4. **Test Thoroughly**: Run complete workflow with various manual scenarios
5. **Validate Outputs**: Ensure SAR documents meet regulatory requirements
6. **Document Results**: Create final project documentation and metrics
7. **Prepare Presentation**: Demonstrate your system's capabilities and business value

**Congratulations on building a complete AI-powered SAR processing system! 🎉**

## 🧪 Step 7: Workflow Testing and Validation

Before finalizing your implementation, validate your complete system with comprehensive testing.

In [ ]:
# 🧪 Workflow Integration Testing
import subprocess

def validate_workflow_components():
    """Validate that all required components are importable"""
    print("🔍 Validating Workflow Components")
    components = {
        'foundation_sar': 'from foundation_sar import CustomerData, CaseData, ExplainabilityLogger, DataLoader',
        'risk_analyst_agent': 'from risk_analyst_agent import RiskAnalystAgent',
        'compliance_officer_agent': 'from compliance_officer_agent import ComplianceOfficerAgent',
        'sar_workflow': 'from sar_workflow import SARWorkflow, HumanReviewGate, screen_customers',
    }
    status = {}
    for name, statement in components.items():
        try:
            exec(statement, {})
            status[name] = True
            print(f"✅ {name} available")
        except ImportError as e:
            status[name] = False
            print(f"❌ {name} not available: {e}")
    return all(status.values())

def run_integration_tests():
    """Run every pre-built and workflow test suite from the project root"""
    suites = {
        'Foundation Components': 'tests/test_foundation.py',
        'Risk Analyst Agent': 'tests/test_risk_analyst.py',
        'Compliance Officer Agent': 'tests/test_compliance_officer.py',
        'Compliance Revision Loop': 'tests/test_compliance_revision.py',
        'Workflow Integration': 'tests/test_workflow.py',
    }
    results = {}
    for label, path in suites.items():
        run = subprocess.run([sys.executable, '-m', 'pytest', path, '-q', '-p', 'no:cacheprovider', '--color=no'],
                             cwd=project_root, capture_output=True, text=True)
        results[label] = (run.returncode == 0, run.stdout.strip().splitlines()[-1])

    print("\n" + "=" * 60)
    print("📊 INTEGRATION TEST RESULTS:")
    for label, (passed, summary) in results.items():
        print(f"   {label:<26} {'✅ PASS' if passed else '❌ FAIL'}  ({summary})")
    all_passed = all(passed for passed, _ in results.values())
    print(f"   Overall Status: {'✅ ALL TESTS PASSED' if all_passed else '❌ SOME TESTS FAILED'}")
    return all_passed

components_ready = validate_workflow_components()
if components_ready:
    print("\n🚀 All components ready - running integration tests")
    integration_passed = run_integration_tests()

In [ ]:
# 🎯 End-to-End Workflow Testing (mocked LLM - no API cost)
import tempfile
from unittest.mock import Mock

def test_complete_workflow():
    """Run the real agents and workflow against a mocked OpenAI client

    Uses two real customers: a structuring case the mock analyst rates High
    (approved at both gates -> SAR filed) and a benign customer rated Low
    (rejected at gate 1 -> no compliance call). Outputs go to a temp directory.
    """
    print("🎯 End-to-End Workflow Testing")

    def mock_llm(**kwargs):
        system_prompt, user_prompt = kwargs['messages'][0]['content'], kwargs['messages'][1]['content']
        response = Mock()
        response.choices = [Mock()]
        response.choices[0].finish_reason = 'stop'
        response.usage = Mock(prompt_tokens=3000, completion_tokens=500, total_tokens=3500)
        if 'Compliance Officer' not in system_prompt:  # the compliance prompt also mentions the Risk Analyst
            suspicious = 'CUST_0053' in user_prompt
            payload = {"classification": "Structuring" if suspicious else "Other",
                       "confidence_score": 0.9, "reasoning": "Profile: ... Decision: ...",
                       "key_indicators": ["7 cash deposits of $9.2k-$9.9k"] if suspicious else ["routine activity"],
                       "risk_level": "High" if suspicious else "Low"}
        else:
            payload = {"narrative": "Tracy Lewis (CUST_0053) made seven cash deposits totalling $67,569.06 between "
                                    "2025-07-25 and 2025-07-31 at five branches, consistent with structuring under 31 USC 5324.",
                       "narrative_reasoning": "Built from the analyst findings.",
                       "regulatory_citations": ["31 CFR 1020.320 (SAR filing requirement)", "31 USC 5324 (Structuring)"],
                       "completeness_check": True}
        response.choices[0].message.content = "Analysis...\n```json\n" + json.dumps(payload) + "\n```"
        return response

    with tempfile.TemporaryDirectory() as tmp:
        mock_client = Mock()
        mock_client.chat.completions.create.side_effect = mock_llm
        test_logger = ExplainabilityLogger(os.path.join(tmp, 'audit.jsonl'))
        test_workflow = SARWorkflow(RiskAnalystAgent(mock_client, test_logger), ComplianceOfficerAgent(mock_client, test_logger),
                                    HumanReviewGate(test_logger, 'e2e_test', mode='simulated_policy'),
                                    DataLoader(test_logger), test_logger, os.path.join(tmp, 'filed_sars'), run_id='E2E')
        queue = [row for row in screening_results.to_dict('records') if row['customer_id'] in ('CUST_0053', 'CUST_0002')]
        records = test_workflow.run(queue, {c['customer_id']: c for c in customers_data}, accounts_data,
                                    transactions_data, progress=None)
        outcome = {r.customer_id: r for r in records}
        filed_files = os.listdir(os.path.join(tmp, 'filed_sars'))
        agent_types = {e['agent_type'] for e in test_logger.entries}

        checks = {
            'Structuring case filed': outcome['CUST_0053'].status == CaseStatus.FILED,
            'Benign case closed at gate 1': outcome['CUST_0002'].status == CaseStatus.REJECTED_AT_DRAFTING,
            'Exactly one SAR document written': len(filed_files) == 1,
            'SAR checksum verifies': verify_sar_document(os.path.join(tmp, 'filed_sars', filed_files[0])),
            'Compliance agent called once (cost gate)': sum(
                'Compliance Officer' in c.kwargs['messages'][0]['content']
                for c in mock_client.chat.completions.create.call_args_list) == 1,
            'Audit trail covers every component': agent_types >= {'DataLoader', 'RiskAnalyst', 'HumanReviewer',
                                                                  'ComplianceOfficer', 'SARFiling', 'Workflow'},
        }
    for check, passed in checks.items():
        print(f"   {'✅' if passed else '❌'} {check}")
    success = all(checks.values())
    print("\n🎉 END-TO-END TEST PASSED!" if success else "\n⚠️ END-TO-END TEST HAD ISSUES")
    return success

test_success = test_complete_workflow()